In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print("Dostępne GPU:", gpus)


In [ ]:
# Ścieżki i parametry
archive_path = '/content/archive.zip'
extract_dir  = '/content/PHCD'
images_dir   = f'{extract_dir}/images'

batch_size = 128
img_height, img_width = 32, 32
validation_split = 0.1
epochs = 40


In [ ]:
# Rozpakowanie ZIP
import os, zipfile

# Stwórz folder, jeśli nie istnieje
os.makedirs(extract_dir, exist_ok=True)

# Rozpakuj, jeśli nie ma jeszcze katalogu images
if not os.path.isdir(images_dir):
    print("Rozpakowuję archive.zip…")
    with zipfile.ZipFile(archive_path, 'r') as z:
        z.extractall(extract_dir)
    print("Rozpakowano do:", images_dir)
else:
    print("Już rozpakowane w:", images_dir)


In [ ]:
# Przygotowanie datasetów
train_ds = tf.keras.utils.image_dataset_from_directory(
    images_dir,
    labels='inferred',
    label_mode='categorical',
    color_mode='grayscale',
    batch_size=batch_size,
    image_size=(img_height, img_width),
    shuffle=True,
    validation_split=validation_split,
    subset='training',
    seed=123
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    images_dir,
    labels='inferred',
    label_mode='categorical',
    color_mode='grayscale',
    batch_size=batch_size,
    image_size=(img_height, img_width),
    shuffle=True,
    validation_split=validation_split,
    subset='validation',
    seed=123
)

class_names = train_ds.class_names
num_classes = len(class_names)
print(f"Znaleziono {num_classes} klas: {class_names}")


In [ ]:
# Optymalizacja pipeline
AUTOTUNE = tf.data.AUTOTUNE

# Cache + prefetch dla szybszego treningu
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.cache().prefetch(buffer_size=AUTOTUNE)


In [ ]:
# Augmentacja i normalizacja
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
])

normalization = tf.keras.layers.Rescaling(1./255)


In [ ]:
# Cell 7: Budowa modelu CNN
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(img_height, img_width, 1)),
    augmentation,
    normalization,

    tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(num_classes, activation='softmax'),
])

model.summary()


In [ ]:
# Kompilacja
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print("Model skompilowany.")


In [ ]:
# Callbacks
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    'phcd_cnn_best.h5', save_best_only=True, monitor='val_accuracy'
)
earlystop_cb = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=5, restore_best_weights=True
)
reducelr_cb = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=3
)

print("Callbacks utworzone.")


In [ ]:
# Trenowanie
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs,
    callbacks=[checkpoint_cb, earlystop_cb, reducelr_cb]
)


In [ ]:
# Ewaluacja i zapis
val_loss, val_acc = model.evaluate(val_ds)
print(f"Dokładność na walidacji: {val_acc:.4f}")

# Zapisujemy finalny model
model.save('phcd_cnn_final.h5')
print("Modele zapisane: phcd_cnn_best.h5 (najlepszy) i phcd_cnn_final.h5 (finalny).")
